<a href="https://colab.research.google.com/github/ahasanulhossain07-del/machine_learning_book/blob/main/Support__Vector_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVC

In [50]:
df=pd.read_csv("/content/titanic_data_updated (3).csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
632,633,yes,first,"Stahelin-Maeglin, Dr. Max",male,32.0,0,0,13214,30.5000,B50,C
65,66,yes,third,"Moubarek, Master. Gerios",male,NaN,1,1,2661,15.2458,NaN,C
768,769,no,third,"Moran, Mr. Daniel J",male,NaN,1,0,371110,24.1500,NaN,Q
876,877,no,third,"Gustafsson, Mr. Alfred Ossian",male,20.0,0,0,7534,9.8458,NaN,S


In [51]:
df['family_size']=df['SibSp']+df['Parch']+1
df['Cabin']=df['Cabin'].fillna('Missing')
df['Deck']=df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,family_size,Deck
741,742,no,first,"Cavendish, Mr. Tyrell William",male,36.0,1,0,19877,78.8500,C46,S,2,C
35,36,no,first,"Holverson, Mr. Alexander Oskar",male,42.0,1,0,113789,52.0000,Missing,S,2,M
654,655,no,third,"Hegarty, Miss. Hanora ""Nora""",female,18.0,0,0,365226,6.7500,Missing,Q,1,M
645,646,yes,first,"Harper, Mr. Henry Sleeper",male,48.0,1,0,PC 17572,76.7292,D33,C,2,D
708,709,yes,first,"Cleaver, Miss. Alice",female,22.0,0,0,113781,151.5500,Missing,S,1,M


In [52]:
X=df.drop(['Survived'],axis=1)
y=df['Survived']

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [53]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,family_size,Deck
331,332,first,"Partner, Mr. Austen",male,45.5,0,0,113043,28.5000,C124,S,1,C
733,734,second,"Berriman, Mr. William John",male,23.0,0,0,28425,13.0000,Missing,S,1,M
382,383,third,"Tikkanen, Mr. Juho",male,32.0,0,0,STON/O 2. 3101293,7.9250,Missing,S,1,M
704,705,third,"Hansen, Mr. Henrik Juul",male,26.0,1,0,350025,7.8542,Missing,S,2,M
813,814,third,"Andersson, Miss. Ebba Iris Alfrida",female,6.0,4,2,347082,31.2750,Missing,S,7,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,107,third,"Salkjelsvik, Miss. Anna Kristine",female,21.0,0,0,343120,7.6500,Missing,S,1,M
270,271,first,"Cairns, Mr. Alexander",male,NaN,0,0,113798,31.0000,Missing,S,1,M
860,861,third,"Hansen, Mr. Claus Peter",male,41.0,2,0,350026,14.1083,Missing,S,3,M
435,436,first,"Carter, Miss. Lucile Polk",female,14.0,1,2,113760,120.0000,B96 B98,S,4,B


In [54]:
y_train

,Survived
331,no
733,no
382,no
704,no
813,no
...,...
106,yes
270,no
860,no
435,yes


In [55]:
#age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

/tmp/ipykernel_2054/987313337.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


In [56]:
# pipeline

# numerical
p1 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

In [57]:
# categories goes one after another via stricly following the order of the input
categories = [['third','second','first']]

In [58]:
# pipeline

#  categorical columns

p3 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())
    ]
)

In [59]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'
)
preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

In [60]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [61]:
SVC_model=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',SVC())
    ]
)

In [62]:
from sklearn.model_selection import GridSearchCV

In [63]:
grid_param =[ {
    "model__kernel" : ['linear'],
    "model__C" :[0.01 , 0.1 , 1 , 10 , 50 , 100]
},
  {
      "model__kernel" : ['rbf'] ,
      "model__C" :[0.01 , 0.1 ,1,100],
      "model__gamma" :[0.01,0.1,5,10,'scale','auto']
  },
    {
        "model__kernel": ['poly'],
        "model__C" :[0.01 , 0.1 , 1 ,100],
        "model__degree" : [2,3]
    }

]


In [64]:
best_SVC_model=GridSearchCV(
    estimator=SVC_model,
    param_grid=grid_param,
    cv=5
)

In [65]:
best_SVC_model.fit(X_train,y_train)

ValueError: 
All the 190 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
190 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Family_Size'

The above exception was the direct cause of the following ex

In [ ]:
y_pred_train = best_SVC_model.predict(X_train)

print(accuracy_score(y_train,y_pred_train))

In [ ]:
y_pred = best_SVC_model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)
print(accuracy)
precision = precision_score(y_test,y_pred)
print(precision)
recall = recall_score(y_test,y_pred)
print(recall)